# 第 5 周 · 周日：周复习——SQL分析思维回顾与综合实战

> 难度：⭐⭐⭐ | 今日目标：系统回顾本周学到的SQL分析技能，完成一个综合SQL任务验证掌握程度
> 配对课程：01python_learn W5D7（周考核）
> 本周主题：SQL分析思维——从取数到分析

---

## 一、本周知识地图——你学到了什么

### 1.1 本周核心：SQL不只是取数——用窗口函数、CTE、JOIN这些工具，直接在SQL里完成分析逻辑

```
D1 SQL分析思维入门：取数 vs 分析，执行顺序，三步设计法
  ↓ 建立分析思维框架
D2 窗口函数：ROW_NUMBER/RANK/DENSE_RANK/LAG/SUM OVER
  ↓ 掌握SQL最强大的分析工具——聚合不折叠
D3 连续登录与留存：Gap-and-Island三步法，留存SQL
  ↓ 解决连续行为分析问题——把"连续"变成"相等"
D4 JOIN全家桶：四种JOIN + 三大陷阱（行数膨胀/WHERE vs ON/NOT IN）
  ↓ 避免最隐蔽的数据错误
D5 CTE与子查询：用WITH把复杂逻辑分层
  ↓ 让SQL可读、可维护、可调试
D6 三大经典SQL：TopN / 复购率 / 漏斗
  ↓ 覆盖70%的日常分析需求
D7 周复习：综合实战 ← 你在这里
```

### 1.2 本周每天一个最重要的认知

| 日期 | 最重要的认知 |
|------|------------|
| D1 | 写SQL前先用30秒想清楚：一行代表什么？从哪些表取？数据怎么变换？ |
| D2 | 窗口函数 = GROUP BY的能力 + 不折叠行——填补了集合语言的根本空白 |
| D3 | gap-and-island的核心是"日期减序号"——把"连续"变成"相等" |
| D4 | LEFT JOIN + WHERE右表条件 = INNER JOIN + NOT IN有NULL陷阱 |
| D5 | CTE让你的分析思路直接体现在SQL结构上——每个WITH子句对应一个步骤 |
| D6 | TopN/复购率/漏斗——先聚合到分析粒度，再计算指标 |

### 1.3 本周技能树

```
SQL分析思维
├── 分析框架：三步设计法（粒度→数据源→变换）
├── 排名与累计：窗口函数（ROW_NUMBER/RANK/LAG/SUM OVER）
├── 连续行为：Gap-and-Island（日期减序号）+
│   Sessionization（时间间隔切分）
├── 多表关联：JOIN（INNER/LEFT/RIGHT/FULL）+
│   陷阱规避（WHERE vs ON/行数膨胀/NOT IN）
├── 代码组织：CTE（链式定义+多次引用+递归）
└── 经典模式：TopN/复购率/漏斗
```

### 1.4 SQL分析的通用方法论

本周学到的方法论不只适用于SQL，它是一种通用的数据分析思维方式：

```
1. 理解业务问题 → 问清楚对方真正想要什么
2. 翻译成数据问题 → 把业务语言变成数据语言（指标定义）
3. 设计分析逻辑 → 确定粒度、数据源、变换步骤
4. 用SQL实现 → 用CTE分步、窗口函数计算、JOIN关联
5. 验证结果 → 抽样检查、总量校验、业务逻辑验证
6. 解读输出 → 不止给数字，给业务解读和建议
```

---

## 二、本周知识点回顾速查表

### 2.1 SQL执行顺序（必须记住！）

| 步骤 | 子句 | 关键点 |
|------|------|--------|
| 1 | FROM + JOIN | 笛卡尔积 + ON过滤 |
| 2 | WHERE | 过滤行（看不到SELECT别名！） |
| 3 | GROUP BY | 分组 |
| 4 | HAVING | 过滤组（可以看到聚合结果） |
| 5 | SELECT | 选择列、命名别名 |
| 6 | ORDER BY | 排序（可以看到别名） |
| 7 | LIMIT | 限制行数 |

### 2.2 窗口函数速查

| 函数 | 用途 | 关键参数 |
|------|------|--------|
| ROW_NUMBER() | 严格唯一排名 | PARTITION BY + ORDER BY |
| RANK() | 跳跃排名 | 同上 |
| DENSE_RANK() | 连续排名 | 同上 |
| LAG(col, n) | 向前取n行 | PARTITION BY + ORDER BY |
| LEAD(col, n) | 向后取n行 | 同上 |
| SUM(col) OVER() | 累计求和 | ORDER BY（定义窗口） |
| AVG(col) OVER() | 移动平均 | ROWS BETWEEN n PRECEDING |
| FIRST_VALUE(col) | 窗口第一个值 | ORDER BY |

### 2.3 JOIN速查

| JOIN类型 | 保留规则 | 何时使用 |
|---------|---------|--------|
| INNER JOIN | 两边都有才保留 | 只关心匹配的数据 |
| LEFT JOIN | 左表全保留 | 找"有但没有"的模式 |
| RIGHT JOIN | 右表全保留 | 同LEFT JOIN（换顺序） |
| FULL JOIN | 两边都全保留 | 找"差异" |
| CROSS JOIN | 笛卡尔积 | 生成组合 |
| ANTI JOIN | LEFT JOIN + IS NULL | 找"左有右没有" |

### 2.4 CTE速查

| 用法 | 语法 |
|------|------|
| 单CTE | WITH name AS (SELECT ...) SELECT * FROM name |
| 多CTE | WITH a AS (...), b AS (... FROM a), c AS (...) SELECT ... FROM c |
| 多次引用 | 同一CTE在后续多个CTE或主查询中引用 |
| 递归CTE | WITH RECURSIVE name AS (锚定 UNION ALL 递归) SELECT ... FROM name |

### 2.5 经典分析SQL速查

| 模式 | 核心SQL |
|------|--------|
| TopN | ROW_NUMBER() OVER(PARTITION BY ... ORDER BY ...) WHERE rn<=N |
| 复购率 | COUNT(CASE WHEN order_cnt>=2 THEN 1 END) / COUNT(*) |
| 漏斗 | COUNT(DISTINCT CASE WHEN event=X THEN user_id END) |
| 连续登录 | DATE(dt, '-'||ROW_NUMBER()||' days') AS grp GROUP BY uid,grp |
| 留存率 | LEFT JOIN ON DATE(base_date, '+'||N||' days') |
| Session | SUM(CASE WHEN gap>30min THEN 1 ELSE 0 END) OVER(... ORDER BY ...) |

### 2.6 常见陷阱速查

| 陷阱 | 症状 | 解决方案 |
|------|------|--------|
| WHERE右表条件 | LEFT JOIN变成INNER JOIN | 右表条件放ON |
| 一对多膨胀 | SUM值偏大N倍 | 先聚合再JOIN或COUNT(DISTINCT) |
| NOT IN + NULL | 返回空集 | NOT EXISTS或LEFT JOIN+IS NULL |
| WHERE用别名 | 报错 | 改用HAVING或重复表达式 |
| 忽略去重 | 留存率被重复登录影响 | 先用DISTINCT或GROUP BY |
| ROWS/RANGE混淆 | 时间序列分析结果错误 | ROWS=物理行，RANGE=值范围 |

---

## 三、周考核：综合SQL任务——用户健康度月报

### 场景描述

你是电商数据分析师，运营总监让你做一个"用户健康度月报"，涵盖三个维度：

1. **用户规模指标**：本月活跃用户数、新用户数、复购用户数、复购率
2. **品类消费Top5**：每个品类的消费Top5用户及其排名
3. **高价值用户识别**：连续购买≥3次（任意时间段内有过3笔订单且相邻两笔间隔不超过7天）的用户列表

请用本周学到的SQL技能（CTE + 窗口函数 + JOIN + 留存思维）写出完整的查询。

### 数据准备

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

# 生成模拟订单数据（500行，100个用户，4个品类，1月全月）
n = 500
df = pd.DataFrame({
    'user_id': np.random.choice(range(1, 101), n),
    'category': np.random.choice(['服装', '数码', '食品', '美妆'], n),
    'amount': np.round(np.random.lognormal(4, 0.7, n), 2),
    'order_date': np.random.choice(pd.date_range('2026-01-01', '2026-01-31'), n)
})
df['order_date'] = df['order_date'].astype(str)
df.to_sql('orders', conn, index=False)

print(f"订单数据: {n} 行")
print(f"用户数: {df['user_id'].nunique()}")
print(f"品类数: {df['category'].nunique()}")
print(f"日期范围: {df['order_date'].min()} ~ {df['order_date'].max()}")
print()
print("开始周考核...")
print()

### 任务1：用户规模指标（活跃用户数/新用户数/复购率）

In [ ]:
print("=" * 60)
print("任务1：本月用户规模指标")
print("=" * 60)

q1 = """WITH
-- 每个用户的订单统计
user_stats AS (
    SELECT user_id,
           COUNT(*) AS order_cnt,
           SUM(amount) AS total_amount,
           MIN(order_date) AS first_order_date,
           MAX(order_date) AS last_order_date
    FROM orders
    GROUP BY user_id
)
-- 整体指标
SELECT
    COUNT(*) AS active_users,
    SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) AS repeat_users,
    ROUND(SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS repurchase_rate,
    ROUND(AVG(order_cnt), 1) AS avg_orders_per_user,
    ROUND(AVG(total_amount), 1) AS avg_amount_per_user
FROM user_stats"""

result1 = pd.read_sql(q1, conn)
print(result1.to_string())
print()
print("解读：")
print(f"  活跃用户: {result1['active_users'].iloc[0]} 人")
print(f"  复购用户: {result1['repeat_users'].iloc[0]} 人")
print(f"  复购率: {result1['repurchase_rate'].iloc[0]}%")
print(f"  人均订单: {result1['avg_orders_per_user'].iloc[0]} 笔")
print(f"  人均消费: {result1['avg_amount_per_user'].iloc[0]} 元")

### 任务2：每个品类消费Top5用户

In [ ]:
print("=" * 60)
print("任务2：每个品类消费Top5用户")
print("=" * 60)

q2 = """SELECT category, user_id,
    ROUND(total_amount, 1) AS total_amount,
    order_cnt,
    rn AS category_rank
FROM (
    SELECT category, user_id,
        SUM(amount) AS total_amount,
        COUNT(*) AS order_cnt,
        ROW_NUMBER() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS rn
    FROM orders
    GROUP BY category, user_id
)
WHERE rn <= 5
ORDER BY category, rn"""

result2 = pd.read_sql(q2, conn)
print(result2.to_string())
print()
print("每个品类Top1用户的消费金额均在该品类中遥遥领先")
print("这些用户是该品类的'超级用户'——可以考虑针对性运营")

### 任务3：连续购买≥3次的用户（相邻间隔≤7天）

In [ ]:
print("=" * 60)
print("任务3：连续购买≥3次（间隔≤7天）的用户")
print("=" * 60)

q3 = """WITH
-- 步骤1：对每个用户的订单按日期排序，计算相邻订单间隔
ordered AS (
    SELECT user_id, order_date, amount,
        LAG(order_date) OVER(PARTITION BY user_id ORDER BY order_date) AS prev_date
    FROM orders
),
-- 步骤2：标记是否开启新的连续段（间隔>7天或首单）
marked AS (
    SELECT user_id, order_date, amount, prev_date,
        CASE
            WHEN prev_date IS NULL THEN 1
            WHEN JULIANDAY(order_date) - JULIANDAY(prev_date) > 7 THEN 1
            ELSE 0
        END AS new_streak
    FROM ordered
),
-- 步骤3：分配连续段编号
streaked AS (
    SELECT user_id, order_date, amount,
        SUM(new_streak) OVER(PARTITION BY user_id ORDER BY order_date) AS streak_id
    FROM marked
),
-- 步骤4：统计每个连续段的长度
streak_stats AS (
    SELECT user_id, streak_id,
        COUNT(*) AS consecutive_orders,
        MIN(order_date) AS streak_start,
        MAX(order_date) AS streak_end,
        SUM(amount) AS streak_total
    FROM streaked
    GROUP BY user_id, streak_id
    HAVING consecutive_orders >= 3
)
-- 步骤5：输出符合条件的用户
SELECT user_id,
    MAX(consecutive_orders) AS max_consecutive,
    COUNT(DISTINCT streak_id) AS streak_count
FROM streak_stats
GROUP BY user_id
ORDER BY max_consecutive DESC"""

result3 = pd.read_sql(q3, conn)
print(result3.to_string())
print()
print("找到连续购买≥3次（间隔≤7天）的用户：")
print(f"  共 {len(result3)} 个用户符合条件")
print(f"  最长连续购买: {result3['max_consecutive'].max()} 笔")
print()
print("这批用户是'高频购买者'——是推送限时活动的最佳目标群体")

### 任务4（附加）：各品类复购率对比

In [ ]:
print("=" * 60)
print("任务4（附加）：各品类复购率对比")
print("=" * 60)

q4 = """WITH user_category AS (
    SELECT user_id, category, COUNT(*) AS order_cnt
    FROM orders
    GROUP BY user_id, category
)
SELECT category,
    COUNT(*) AS buyers,
    SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) AS repeat_buyers,
    ROUND(SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS repurchase_rate,
    ROUND(AVG(order_cnt), 1) AS avg_orders_per_buyer
FROM user_category
GROUP BY category
ORDER BY repurchase_rate DESC"""
print(pd.read_sql(q4, conn))
print()
print("解读：不同品类的复购率反映品类的消费属性")
print("  食品复购率通常最高（快消品，消费频次高）")
print("  数码复购率通常最低（耐用品，更新周期长）")

---

## 四、知识体系整合——SQL分析的"万能公式"

通过本周的学习，我们可以总结出一个SQL分析的"万能公式"：

```
WITH
-- 1. 数据准备：过滤、去重、清洗
clean_data AS (
    SELECT ... FROM raw_table WHERE ... GROUP BY ...
),
-- 2. 粒度聚合：计算基础指标
aggregated AS (
    SELECT ... FROM clean_data
    [LEFT JOIN ...]
    GROUP BY ...
),
-- 3. 窗口计算：排名、累计、环比
windowed AS (
    SELECT *,
        ROW_NUMBER() / LAG() / SUM() OVER(PARTITION BY ... ORDER BY ...)
    FROM aggregated
),
-- 4. 变换分群：CASE WHEN打标签
labeled AS (
    SELECT *,
        CASE WHEN ... THEN 'A' WHEN ... THEN 'B' END AS segment
    FROM windowed
),
-- 5. 最终输出：汇总、排序、限制
SELECT ... FROM labeled
WHERE ...
GROUP BY ...
ORDER BY ...
LIMIT ...;
```

这个公式可以应对80%以上的业务分析需求。当你拿到一个新需求时，在脑里过一遍这五步：
1. 数据怎么清洗？（去重、过滤时间范围）
2. 聚合到什么粒度？（一行代表什么？）
3. 需要什么窗口计算？（排名？累计？环比？）
4. 需要什么变换？（CASE WHEN分群）
5. 最终输出什么？（汇总→排序→限制）

### 跨工具思维映射

| SQL概念 | Pandas | Excel | 核心思维 |
|---------|--------|-------|---------|
| WHERE | df[df['col']>100] | 筛选 | 过滤 |
| GROUP BY | df.groupby() | 数据透视表 | 聚合 |
| CASE WHEN | np.where()/np.select() | IF函数 | 变换 |
| 窗口函数 | df.rolling()/.shift() | 滚动平均/OFFSET | 行级上下文 |
| JOIN | pd.merge() | VLOOKUP/XLOOKUP | 关联 |
| CTE | 中间变量赋值 | 辅助列 | 分步骤 |
| ORDER BY+LIMIT | df.sort_values().head() | 排序+筛选 | TopN |

---

## 五、自我评估清单

请逐项自评（Y=能独立完成，N=需要复习，P=能完成但需要查笔记）：

### 基础能力自评
- [ ] 能写出含PARTITION BY + ORDER BY的窗口函数（ROW_NUMBER/RANK/LAG）
- [ ] 能区分ROW_NUMBER/RANK/DENSE_RANK在并列值时的差异
- [ ] 能说出SQL的执行顺序（FROM→WHERE→GROUP BY→HAVING→SELECT→ORDER BY）
- [ ] 能写出四种JOIN的韦恩图
- [ ] 能用CTE改写含子查询的SQL

### 中级能力自评
- [ ] 能用gap-and-island三步法找出连续N天登录的用户
- [ ] 能写出次日留存和7日留存的SQL
- [ ] 能解释LEFT JOIN + WHERE右表条件为什么=INNER JOIN
- [ ] 能说出NOT IN的NULL陷阱和替代方案
- [ ] 能分析一对多JOIN的行数膨胀是否合理
- [ ] 能写出递归CTE（锚定成员+递归成员）

### 高级能力自评
- [ ] 能写出TopN/复购率/漏斗的完整SQL
- [ ] 能把一个业务问题完整翻译成CTE组织的SQL
- [ ] 能组合窗口函数+CTE+JOIN解决复杂分析问题
- [ ] 能从业务方的"取数需求"中识别出真正的分析需求
- [ ] 能评估一个SQL的性能（扫描量、JOIN大小、倾斜风险）

### 面试准备自评
- [ ] "窗口函数和GROUP BY的区别？"
- [ ] "ROW_NUMBER/RANK/DENSE_RANK的区别？"
- [ ] "如何计算连续登录N天？"
- [ ] "INNER JOIN和LEFT JOIN的区别？为什么LEFT JOIN+WHERE会变成INNER JOIN？"
- [ ] "NOT IN和NOT EXISTS的区别？"
- [ ] "CTE和子查询的区别？什么时候用CTE？"
- [ ] "如何取每组TopN？"
- [ ] "复购率怎么算？漏斗分析的SQL怎么写？"

---

## 六、跨模块连接与前瞻

### 本周在整体学习路线中的位置

```
第4周：Python数据分析 + 数仓分层
   ↓
第5周：SQL分析思维（上）← 你刚完成
   ├── SQL分析思维框架
   ├── 窗口函数与连续行为
   ├── JOIN与多表操作
   ├── CTE与代码组织
   └── 经典分析SQL模式
   ↓
第6周：SQL分析思维（下）← 下周开始
   ├── 多维拆解分析
   ├── 复杂业务逻辑SQL
   └── SQL性能调优进阶
```

### 与01python_learn的协同

| 01python_learn | 03analyst_thinking | 协同效果 |
|---------------|-------------------|--------|
| W5D1-7 Python进阶 | W5D1-7 SQL分析思维 | Python与SQL思维互补 |
| W6 SQL基础语法 | W6 SQL分析（下） | 01学语法，03学思维 |
| W7 Python数据分析 | W7 综合案例 | 用Pandas验证SQL思路 |

### 扩展阅读建议

以下02analyst_thinking的资源可以作为深入学习：
- W1D1: 窗口函数完整知识图谱（含面试题）
- W1D2: 连续登录与留存（含大数据优化）
- W1D3: JOIN全家桶（含执行计划分析）
- W1D4: CTE与子查询（含性能对比）
- W1D5: 日期函数与时间序列
- W1D6: 字符串函数与正则表达式
- W1D7: SQL性能优化

---

## 七、本周总结

> **本周核心收获**：SQL分析思维的建立——从"别人叫我取数我就取数"到"我理解业务问题，用SQL直接给出答案"。你学到了：三步设计法（粒度→数据源→变换）、窗口函数（聚合不折叠）、gap-and-island（把连续变成相等）、JOIN陷阱（WHERE vs ON / 行数膨胀 / NOT IN NULL）、CTE分步（让复杂SQL可读）、三大经典模式（TopN/复购率/漏斗）。这些都是你作为数据分析师的核心竞争力——不是会不会写SQL，而是能不能用SQL解决业务问题。

**三条金句带回家**：
1. 写SQL前先用30秒想清楚三件事——输出粒度、数据源、变换逻辑——这30秒帮你省下30分钟
2. SQL是一门分析语言，不只是取数工具——你的价值在于把业务问题翻译成SQL，输出答案而非数据
3. 窗口函数+CTE+JOIN是SQL分析的三驾马车——掌握这三者，你就掌握了80%的SQL分析能力

### 课后拓展

1. **实战**：找一个你工作中的实际业务问题，用本周学的"万能公式"从头到尾写出完整的分析SQL

2. **代码审查**：找一个你之前写的复杂SQL（超过2层嵌套），用CTE重构它——对比重构前后的可读性

3. **性能挑战**：如果订单表有10亿行，今天写的月报SQL有哪些地方需要优化？（提示：分区裁剪、索引设计、数据倾斜）

4. **模拟面试**：找同事或对着镜子，完整回答一遍本周所有的"面试准备"问题

---

*03analyst_thinking · Week 5 Day 7 · 2026-05-21*

> **下周见！第6周我们将继续SQL分析思维（下），进入更复杂的多维拆解和性能调优。届时01python_learn W6也会同步开始SQL基础语法——01教你"怎么写"，03教你"为什么这么写"和"还能怎么写更好"。**